# Aporte adicional: RAG + LLM

Este notebook es adicional al pipeline de carga y Power BI. Pobla la base vectorial del RAG desde `predicciones_riesgo` y permite hacer consultas a la base de datos en lenguaje natural.

In [ ]:
import sys
import os
import importlib
import logging
import subprocess
import warnings
from pathlib import Path
from IPython.utils import io as _ipio

current_dir = Path(os.getcwd())
ROOT_DIR = current_dir if (current_dir / "src").exists() else current_dir.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import src.config as _cfg_module
importlib.reload(_cfg_module)
from src.config import settings
from src.db_utils import create_supabase_engine, get_database_host

engine = create_supabase_engine(settings.DATABASE_URL)
print(f"Base de datos activa: {get_database_host(settings.DATABASE_URL)}")

In [ ]:
os.environ["TQDM_DISABLE"] = "1"
warnings.filterwarnings("ignore")
for library_name in [
    "httpx", "httpcore", "sentence_transformers", "transformers",
    "huggingface_hub", "langchain", "langchain_core",
    "langchain_community", "src.rag_agent", "openai",
]:
    logging.getLogger(library_name).setLevel(logging.ERROR)

print("Poblando RAG knowledge base...", end=" ", flush=True)
with _ipio.capture_output():
    result = subprocess.run(
        [sys.executable, "scripts/populate_rag.py"],
        capture_output=True,
        text=True,
        env={**os.environ, "TQDM_DISABLE": "1"},
    )

if result.returncode != 0:
    print(f"Error codigo {result.returncode}\n{result.stderr[-400:]}")
else:
    for line in reversed((result.stdout + result.stderr).splitlines()):
        if "chunks" in line or "RAG Knowledge Base" in line:
            print(line.split(" - ")[-1])
            break

from src.rag_agent import RAGAgentPipeline

agent = RAGAgentPipeline(
    db_connection=settings.DATABASE_URL,
    nvidia_api_key=settings.NVIDIA_API_KEY,
)

pregunta = "Cual es la tasa de mora actual del portafolio?"
print(f"Usuario: {pregunta}")
print("Asesor: ", end="", flush=True)
for chunk in agent.stream_query(pregunta):
    print(chunk, end="", flush=True)